# Salt Surface Forcing

Salt Surface Forcing from Surface Salinity Change in the MEOP Profiles.

##  Import Packages

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import xmitgcm.utils
import gsw as gsw
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

import mat73 # for loading .mat files version >= 7.3

## Load Data

In [2]:
filepath = "/isibhv/projects-noreplica/p_so-clim/alhaum001/supercool_profiles_merged_20200911.mat"
matstructs= mat73.loadmat(filepath) # this loads the matlab file as a python dictionary
matstructs.keys() # the dictionary keys (the names of the mat structures in the .mat file)

dict_keys(['data0', 'data1', 'data1_other', 'data2'])

In [3]:
data1 = matstructs['data1']
data1_other = matstructs['data1_other']
data2 = matstructs['data2']

- data1: metadata of the supercooled profiles
- data1_other: metadata non-supercooled profiles
- data2: the supercooled profiles

### Add Datetime

In [4]:
# add datetime variable
zero_time = datetime(1,1,1) # Matlab's day zero is 0000-01-01, but Python's datetime starts at 0001-01-01, so we need to account for that when converting

# DATA1
days = data1["dates"]-1-366 # convert from Matlab's datenum to number of days since 0001-01-01, accounting for the fact that Matlab's datenum starts with 1 at 0000-01-01 and includes a leap year in year 0
datetimes = [
    zero_time + timedelta(days=day) if day == day else float('nan') # convert to datetime, respecting NaN values
    for day in days
]

data1["datetimes"] = pd.Series(np.array(datetimes))

# DATA1_OTHER
days_other = data1_other["dates"]-1-366
datetimes_other = [
    zero_time + timedelta(days=day) if day == day else float('nan')
    for day in days_other
]

data1_other["datetimes"] = pd.Series(np.array(datetimes_other))


### Filter for Seal 9900109

In [13]:
mask_9900109 = (data1["id"] == 9900109)

data1_9900109 = {key: data1[key][mask_9900109] for key in data1.keys()}
data2_9900109 = {key: data2[key][:,mask_9900109] for key in data2.keys()}

mask_other_9900109 = (data1_other["id"] == 9900109)
data1_other_9900109 = {key: data1_other[key][mask_other_9900109] for key in data1_other.keys()}

## Select Cluster

Select July cluster from 2008-07-16 08 to 2008-07-19 12

In [57]:
start = datetime(2008,7,16,8)
end = datetime(2008,7,19,12)

mask = (data1_9900109['datetimes'] > start) & (data1_9900109['datetimes'] < end)

data1_cluster = pd.DataFrame(data1_9900109)[mask]

data2_cluster = {key: data2_9900109[key][:, mask]
     for key in ["dpth",'dpth_mid',"s"]} 
data2_cluster

{'dpth': array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(1050, 10)),
 'dpth_mid': array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(1049, 10)),
 's': array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(1050, 10))}

## Calculate Salt Flux

In [65]:
print(f"Minimum layer thickness: {np.nanmin(data2_cluster['dpth_mid'][1:,:]-data2_cluster['dpth_mid'][:-1,:]):.2f}")
print(f"Maximum layer thickness: {np.nanmax(data2_cluster['dpth_mid'][1:,:]-data2_cluster['dpth_mid'][:-1,:]):.2f}")

Minimum layer thickness: 0.99
Maximum layer thickness: 0.99


In [133]:
# Salt Flux based on whole profile depth
s_mean = np.nanmean(data2_cluster['s'], axis=0) # unit PSU, similar to g/kg
max_depth = np.nanmax(data2_cluster["dpth"])
total_salt = s_mean * max_depth * 1000# water column salt content g/m^2
total_salt_change = total_salt[1:]-total_salt[0:-1] # g/m^2/delta t
time_diff = (cluster_data1["dates"].values[1:]-cluster_data1["dates"].values[0:-1])*24*3600 # delta t in s

print("Salt flux between profiles based on whole 800m deep profile (g/(m^2s)):")
print(*np.round(total_salt_change/time_diff,2))

Salt flux between profiles based on whole 800m deep profile (g/(m^2s)):
-0.14 0.02 1.7 -0.29 -0.88 0.12 -3.47 -5.33 7.0


In [140]:
# Salt Flux based on upper XXXm
# to be improved: PSU->g/kg; water density
max_depth = 130
s_mean = np.nanmean(data2_cluster['s'][0:max_depth,:], axis=0) # unit PSU, similar to g/kg
total_salt = s_mean * max_depth * 1000# water column salt content g/m^2
total_salt_change = total_salt[1:]-total_salt[0:-1] # g/m^2/delta t
time_diff = (cluster_data1["dates"].values[1:]-cluster_data1["dates"].values[0:-1])*24*3600 # delta t in s

print(f"Salt flux between profiles based on upper {max_depth}m [g/(m^2 s)]:")
print(*np.round(total_salt_change/time_diff,2))
print("Mean Value:")
print(f"{np.mean(total_salt_change/time_diff):.2f}")

Salt flux between profiles based on upper 130m [g/(m^2 s)]:
-0.02 -0.06 0.61 -0.3 -0.22 0.07 0.1 -0.3 0.35
Mean Value:
0.03


=> Let's take 0.03 g/(m^2 s) as a starting point.

## Write Binary

In [150]:
salt_flux_val = -0.03
salt_flux_const = np.zeros((36,66,594))
salt_flux_const[:,:,285:310] = salt_flux_val

# write binary
out_filename = "bin_forc_SA_36x66x594_100m_lead_const_-0.03.bin"
xmitgcm.utils.write_to_binary(salt_flux_const.flatten(order='C'), '../../MITgcm/so_plumes/input/' + out_filename)

In [ ]:
rowans_lead_name = "bin_forc_Q_36x66x594_100m_lead"

## Compare with Rowan's

In [141]:
# import Rowan's forcing field
rowans_lead_name = "bin_forc_SA_36x66x594_100m_lead_037"
r_filename = rowans_lead_name + ".bin"
rowans_salt_flux= xmitgcm.utils.read_raw_data('../../MITgcm/so_plumes/input/' + r_filename, shape=(36,66,594), dtype=np.dtype('>f4') ) # shape is (time, X, Y)

## MIZ Test

In [7]:
Nx = 50 # east-west
Ny = 100 # north south
t = 36 # forcing period in h
ice_edge = int(Ny/2) # location of ice edge
S_const = 0

S_MIZ = np.zeros((Nx,Ny,t)) # if flattening in F-order this should be the correct shape
S_MIZ[:,0:ice_edge,:] = S_const

# write binary
out_filename = "bin_forc_SA_" + f"Nx{Nx}_Ny{Ny}_t{t}_0.bin"
xmitgcm.utils.write_to_binary(S_MIZ.flatten(order='F'), '../../MITgcm/so_plumes/input/' + out_filename)
print(out_filename)

bin_forc_SA_Nx50_Ny100_t36_0.bin
